# HW2: Evaluation of TFLite Accuracy & Size

Use this notebook to measure TFLite and Accuracy of your model.

### Import the required modules

In [1]:
import tensorflow as tf
import numpy as np
import os
import zipfile
from glob import glob

2024-12-17 16:13:02.688170: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-17 16:13:02.688223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-17 16:13:02.689432: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-17 16:13:02.697569: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-17 16:13:03.707545: W tensorflow/compiler/tf2

### Set Model & Hyperparameters

**You may only modify the following cell by setting the values for `MODEL_FILE_NAME` and `PREPROCESSING_ARGS`. Ensure that the `PREPROCESSING_ARGS` values match those used during training. No other modifications are permitted in this notebook.**

In [2]:
#MODEL_FILE_PATH = 'tflite_models/1730886043.tflite.zip' # file extension can be .tflite or .zip
MODEL_FILE_PATH = 'tflite_models/first_model.tflite' 

PREPROCESSING_ARGS = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.008,
    'frame_step_in_s': 0.002,
    'num_mel_bins': 40,
    'lower_frequency': 20,
    'upper_frequency': 4000,
    'num_coefficients': 0  # set num_coefficients to 0 if log-Mel Spectrogram features have been used
}

LABELS = ['down', 'up']

### Instantiate Pre-processing

In [3]:
from reader import AudioReader
from preprocessing import Padding, Normalization
from preprocessing import MelSpectrogram, MFCC


audio_reader = AudioReader(tf.int16)

padding = Padding(PREPROCESSING_ARGS['sampling_rate'])
normalization = Normalization(tf.int16)


if PREPROCESSING_ARGS['num_coefficients'] == 0:
    PREPROCESSING_ARGS.pop('num_coefficients')
    feature_processor = MelSpectrogram(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mel_spec
else:
    feature_processor = MFCC(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mfccs

2024-12-17 16:13:04.892379: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-12-17 16:13:04.959343: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-12-17 16:13:04.962021: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

### Measure TFLite Model Size

In [4]:
model_size = os.path.getsize(MODEL_FILE_PATH)

FileNotFoundError: [Errno 2] No such file or directory: 'tflite_models/first_model.tflite'

### Load the TFLite Model

In [5]:
if MODEL_FILE_PATH.endswith('.zip'):
    with zipfile.ZipFile(MODEL_FILE_PATH, 'r') as fp:
        fp.extractall('/tmp/')
        model_filename = fp.namelist()[0]
        MODEL_FILE_PATH = '/tmp/' + model_filename

interpreter = tf.lite.Interpreter(model_path=MODEL_FILE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Number of inputs:", len(input_details))
print("Number of outputs:", len(output_details))
print("Input name:", input_details[0]['name'])
print("Input shape:", input_details[0]['shape'])
print("Output name:", output_details[0]['name'])
print("Output shape:", output_details[0]['shape'])

Number of inputs: 1
Number of outputs: 1
Input name: serving_default_input_1:0
Input shape: [  1 499  40   1]
Output name: StatefulPartitionedCall:0
Output shape: [1 2]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


### Test the TFLite Model Accuracy

In [7]:
SCRIPT_DIR = os.path.abspath('')

filenames = glob(os.path.join(SCRIPT_DIR, 'msc-train/down*')) + glob(os.path.join(SCRIPT_DIR, 'msc-train/up*'))

accuracy = 0.0

for filename in filenames:
    audio, true_label = audio_reader.get_audio_and_label(filename)   
    true_label = true_label.numpy().decode()
    
    audio = padding.pad_audio(audio)
    audio = normalization.normalize_audio(audio)
    features = feature_processor_fn(audio)
    features = tf.expand_dims(features, 0)
    features = tf.expand_dims(features, -1)

    interpreter.set_tensor(input_details[0]['index'], features)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    top_index = np.argmax(output[0])
    predicted_label = LABELS[top_index]

    accuracy += true_label == predicted_label

accuracy /= len(filenames)

### Report the Test Results

In [8]:
print(f'Accuracy: {100 * accuracy:.3f}%')
print(f'Model size: {model_size / 2 ** 10:.1f}KB')

Accuracy: 90.062%
Model size: 548.8KB


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ab5228b8-e3a2-4c2f-9e86-8cb52d8a840d' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>